# GRU-D — Student Exercise

Build a **GRU-D** survival model that predicts 5-year lung-cancer risk from
irregular longitudinal lab measurements.

GRU-D extends a vanilla GRU by learning *exponential decay*: the longer a
lab value has been unobserved, the more the model forgets it and reverts to
the population mean. This makes it well-suited to EHR data where labs are
measured sporadically and at irregular intervals.

> Che et al. (2018) *Recurrent Neural Networks for Multivariate Time Series
> with Missing Values*, Scientific Reports.
> https://doi.org/10.1038/s41598-018-24271-9

## Your task
There are **5 TODOs** — all in `TimeAwareGRUD.forward()` (TODOs 1–4) and
the training loop (TODO 5).  Everything else is complete.

Work through the cells in order, run each one, fill in the TODOs, and aim for:

| Metric | Target |
|---|---|
| Val C-index at landmark 55 | > 0.65 (random ≈ 0.50, production ≈ 0.80) |
| Training loss | decreasing each epoch |

In [ ]:
# ── Google Colab: mount Drive and install missing deps ───────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q lifelines

In [ ]:
import copy
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from lifelines.utils import concordance_index
from torch.utils.data import DataLoader, Dataset

# ── Paths (Google Drive) ─────────────────────────────────────────────────────
base_path = Path('/content/drive/MyDrive/TFM/us-mulcia-2026-ramon/data_tmp')
DATA_DIR  = base_path
OUT_DIR   = base_path / "results" / "gru_d_student"

# ── Hyperparameters ───────────────────────────────────────────────────────────
LANDMARK_AGE      = 55.0   # stand here, look HORIZON years forward
INDEX_AGE         = 18.0   # cohort entry age (patients enter at 18, landmark analysis starts at 45)
HORIZON           = 5.0    # prediction horizon (years)
MAX_SEQ_LEN       = 200    # truncate long sequences to save memory

# Model
HIDDEN_DIM        = 32
EMBED_DIM         = 0      # ICD-10 embedding dim; 0 = disabled (extension exercise)

# Training
BATCH_SIZE        = 256
MAX_EPOCHS        = 30
PATIENCE          = 5
LEARNING_RATE     = 5e-4
WEIGHT_DECAY      = 1e-4
AGE_WEIGHT_GAMMA  = 1.0    # up-weight later-landmark samples slightly
EVENT_OVERSAMPLE  = 2.0    # WeightedRandomSampler multiplier for event patients

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 1. Data

Each pickle contains three items:

| Key | Type | Description |
|---|---|---|
| `"sequences"` | `dict[str, dict]` | one entry per patient |
| `"lab_codes"` | `list[str]` | LOINC codes, one per column of `labs` |
| `"cohort"` | `DataFrame` | `Id, event, duration, status` |

Each patient sequence has:

| Field | Shape | Notes |
|---|---|---|
| `"times"` | `(T,)` | patient age at each observation (years) |
| `"labs"` | `(T, 23)` | z-score-normalised lab values; **0 where not measured** |
| `"masks"` | `(T, 23)` | 1 if measured at this timestep, else 0 |
| `"codes"` | `list[list[str]]` | ICD-10 codes recorded at each timestep (extension) |
| `"sex_male"` | `float` | 0 or 1 |

`duration` = years from cohort entry (age 18) to event or censoring.  
`event` = 1 if lung cancer within the horizon, else 0.

In [ ]:
def load_sequences(pkl_path: Path) -> tuple[dict, list[str], object]:
    with open(pkl_path, "rb") as f:
        cache = pickle.load(f)
    sequences = {pid: seq for pid, seq in cache["sequences"].items() if seq is not None}
    return sequences, list(cache["lab_codes"]), cache["cohort"]


sequences_train, lab_codes, cohort_train = load_sequences(DATA_DIR / "sequences_train.pkl")
sequences_val,   _,         cohort_val   = load_sequences(DATA_DIR / "sequences_val.pkl")

lab_dim = len(lab_codes)
print(f"{lab_dim} lab features")
print(f"Train: {len(sequences_train)} patients  |  event_rate={cohort_train['event'].mean():.3f}")
print(f"Val:   {len(sequences_val)} patients    |  event_rate={cohort_val['event'].mean():.3f}")

# After z-score normalisation the population mean is 0 by construction
mean_x = np.zeros(lab_dim, dtype=np.float32)

23 lab features
Train: 2500 patients  |  event_rate=0.036
Val:   1508 patients    |  event_rate=0.036


## 2. Dataset

`RawSequenceDataset` creates one training sample per (patient, landmark) pair.
For each landmark age, only patients still alive at that age are included;
their sequence is trimmed to observations up to the landmark.

`RawSequenceDataset` creates one sample per (patient, landmark) pair.
ICD-10 codes are loaded but not passed to the model in the basic version
(see the Extensions and Future Work sections for how to add them).
Event oversampling is handled in the training loop via `WeightedRandomSampler`.

In [ ]:
class RawSequenceDataset(Dataset):
    """
    One sample per (patient, landmark_age) pair.
    No ICD embedding in the basic forward pass (extension exercise).
    """

    def __init__(
        self,
        sequences: dict,
        cohort_df,
        landmark_ages: list[float],
        index_age: float,
        max_events: int = 200,
        age_weight_gamma: float = 1.0,
    ):
        self.samples     = []
        self.event_flags = []
        self.lab_dim     = None

        lm_arr = np.array(landmark_ages, dtype=np.float32)
        min_lm, max_lm = float(lm_arr.min()), float(lm_arr.max())

        for row in cohort_df.itertuples(index=False):
            pid      = str(row.Id)
            if pid not in sequences:
                continue
            duration = float(row.duration)
            event    = float(row.event)
            seq      = sequences[pid]
            times    = np.asarray(seq["times"],  dtype=np.float32)
            labs     = np.asarray(seq["labs"],   dtype=np.float32)
            masks    = np.asarray(seq["masks"],  dtype=np.float32)
            sex_male = float(seq["sex_male"])

            if self.lab_dim is None:
                self.lab_dim = labs.shape[1]

            for lm in landmark_ages:
                time_from_lm = duration - (lm - index_age)
                if time_from_lm <= 0:
                    continue

                keep = times <= lm
                if not np.any(keep):
                    times_s = np.array([lm], dtype=np.float32)
                    labs_s  = np.zeros((1, self.lab_dim), dtype=np.float32)
                    masks_s = np.zeros((1, self.lab_dim), dtype=np.float32)
                else:
                    times_s = times[keep]
                    labs_s  = labs[keep]
                    masks_s = masks[keep]

                if max_events and len(times_s) > max_events:
                    times_s = times_s[-max_events:]
                    labs_s  = labs_s[-max_events:]
                    masks_s = masks_s[-max_events:]

                time_since_index_s = times_s - float(index_age)

                # Age-based sample weight (same formula as production)
                lm_norm = (lm - min_lm) / max(max_lm - min_lm, 1e-6)
                weight  = 1.0 + age_weight_gamma * lm_norm

                self.samples.append({
                    "times":            times_s,
                    "labs":             labs_s,
                    "masks":            masks_s,
                    "sex_male":         sex_male,
                    "time_since_index": time_since_index_s,
                    "duration":         time_from_lm,
                    "event":            event,
                    "landmark_age":     float(lm),
                    "weight":           weight,
                    "pid":              pid,
                })
                self.event_flags.append(int(event))

        if self.lab_dim is None:
            self.lab_dim = 0
        n = len(self.samples)
        self.event_rate   = sum(self.event_flags) / max(n, 1)
        self.mean_seq_len = float(np.mean([len(s["times"]) for s in self.samples])) if n else 0.0

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return (
            torch.tensor(s["times"],            dtype=torch.float32),
            torch.tensor(s["labs"],             dtype=torch.float32),
            torch.tensor(s["masks"],            dtype=torch.float32),
            torch.tensor(s["sex_male"],         dtype=torch.float32),
            torch.tensor(s["time_since_index"], dtype=torch.float32),
            torch.tensor(s["duration"],         dtype=torch.float32),
            torch.tensor(s["event"],            dtype=torch.float32),
            torch.tensor(s["landmark_age"],     dtype=torch.float32),
            torch.tensor(s["weight"],           dtype=torch.float32),
            s["pid"],
        )


# Build datasets for a single landmark
train_dataset = RawSequenceDataset(
    sequences_train, cohort_train,
    landmark_ages=[LANDMARK_AGE],
    # landmark_ages=list(range(45,71)), # Para la extension
    index_age=INDEX_AGE,
    max_events=MAX_SEQ_LEN,
    age_weight_gamma=AGE_WEIGHT_GAMMA,
)
val_dataset = RawSequenceDataset(
    sequences_val, cohort_val,
    landmark_ages=[LANDMARK_AGE],
    index_age=INDEX_AGE,
    max_events=MAX_SEQ_LEN,
    age_weight_gamma=AGE_WEIGHT_GAMMA,
)
print(f"Landmark {LANDMARK_AGE}:")
print(f"  train N={len(train_dataset)}  event_rate={train_dataset.event_rate:.3f}  mean_len={train_dataset.mean_seq_len:.1f}")
print(f"  val   N={len(val_dataset)}    event_rate={val_dataset.event_rate:.3f}")

Landmark 55.0:
  train N=41863  event_rate=0.040  mean_len=4.5
  val   N=1028    event_rate=0.042


In [ ]:
def raw_pad_collate(batch):
    """
    Collate variable-length sequences into padded tensors.

    Computes delta_t: the inter-observation gap in years.
    GRU-D uses delta_t to decide how much to decay the last observed value.
    ICD codes are not handled in the basic version (extension exercise).
    """
    times, labs, masks, sex_male, time_since_index, durations, events, landmarks, weights, ids = zip(*batch)

    lengths = torch.tensor([len(t) for t in times])

    padded_labs             = nn.utils.rnn.pad_sequence(labs,             batch_first=True, padding_value=0.0)
    padded_masks            = nn.utils.rnn.pad_sequence(masks,            batch_first=True, padding_value=0.0)
    padded_times            = nn.utils.rnn.pad_sequence(times,            batch_first=True, padding_value=0.0)
    padded_time_since_index = nn.utils.rnn.pad_sequence(time_since_index, batch_first=True, padding_value=0.0)

    # delta_t[0] = 0, delta_t[t] = times[t] - times[t-1]
    deltas = torch.zeros_like(padded_times)
    deltas[:, 1:] = torch.clamp(padded_times[:, 1:] - padded_times[:, :-1], min=0.0)

    return (
        padded_labs,            # x        (B, T, D)
        padded_masks,           # mask     (B, T, D)
        deltas,                 # delta_t  (B, T)
        lengths,                # lengths  (B,)
        torch.stack(sex_male),
        padded_time_since_index,
        torch.stack(durations),
        torch.stack(events),
        torch.stack(landmarks),
        torch.stack(weights),
        list(ids),
    )

## 3. GRU-D Architecture *(TODOs 1–4)*

At each timestep $t$ the model receives:

- $x_t \in \mathbb{R}^D$ — lab values (0 where not measured this timestep)
- $m_t \in \{0,1\}^D$ — binary observation mask
- $\delta_t \in \mathbb{R}$ — years since the previous observation

GRU-D adds **four modifications** to a standard GRU cell — described in
Che et al. (2018), Sections 3.1–3.2. Read those sections and think through
the questions below before looking at the code.

**Step 1 — Lab decay** (Eq. 3, TODO 1)
How should you discount a lab value that was measured $\delta_t$ years ago?
You need a function of $\delta_t$ that equals 1 when $\delta_t = 0$ (just
observed, trust fully) and approaches 0 as $\delta_t \to \infty$ (revert to
the population mean). What function has this shape?

**Respuesta:** La función exponencial negativa cumple con estas condiciones. La idea sería elegir un punto de la recta que une el valor actual obeservado y la media de la población. Es decir, $\hat{x_t}=\gamma_x x_t + (1-\gamma_x) \overline{x}_t$, siendo $\gamma_x=exp(f(\delta_t))$

**Step 2 — Hidden-state decay** (Eq. 4, TODO 2)
Same idea, applied to the hidden state $h_{t-1}$. Why should the model also
partially forget its own state when a long time has passed?

**Respuesta:** La idea sería muy similar, y la razón de este decaimento se basa en una de las premisas sobre las que se sostenta y crea el modelo GRU-D, la homeostasis y la pérdida de influencia en el tiempo. Basicamente, la influencia de una variable observada en el pasado sobre las predicciones actuales disminuye gradualmente a medida que pasa el tiempo sin que se registre una nueva medición. Matematicamente, habría que modelar un parámetro $\gamma_h$, función exponencial negativa para después penalizar con este parámetro el hidden state anterior:
$$
 \hat{h}_{t-1} = \gamma_h h_{t-1}
$$

**Step 3 — Impute missing values** (Eq. 5, TODO 3)
When a lab is *not* observed at time $t$, what should the GRU cell receive?
Think about two cases: (a) the lab was observed — use the real value;
(b) the lab was missing — blend the decayed last observation toward the
population mean. As the decay factor approaches 0, what should the value
converge to?

**Step 4 — Decay the hidden state** (Eq. 6, TODO 4)
Apply the decay from Step 2 to $h_{t-1}$ *before* the GRU cell update.

**Step 5 — GRU cell** (standard — already implemented, do not change)

$$h_t = \operatorname{GRUCell}\bigl([\hat{x}_t \,|\, m_t \,|\, s_i \,|\, \tau_t],\; h_t\bigr)$$

where $s_i$ is sex and $\tau_t$ is time-since-index-age.

The scalar log-risk score is a linear projection of the final hidden state $h_T$.

In [ ]:
class TimeAwareGRUD(nn.Module):
    """
    GRU-D: GRU with temporal decay for irregular, partially-observed time series.

    forward() returns (last_risk, all_risks, h_states):
    - last_risk   (B,)      — risk score at each patient's final observed timestep
    - all_risks   (B, T)    — risk score at every timestep
    - h_states    (B, T, H) — hidden states at every timestep

    flat_codes/offsets are optional — only used in the ICD extension exercise.
    """

    def __init__(
        self,
        input_dim:  int,
        vocab_size: int   = 0,
        mean_x:     np.ndarray | None = None,
        embed_dim:  int   = 0,
        hidden_dim: int   = 32,
        dropout:    float = 0.2,
    ):
        super().__init__()
        self.input_dim  = input_dim
        self.hidden_dim = hidden_dim
        self.embed_dim  = embed_dim

        if mean_x is None:
            mean_x = np.zeros(input_dim, dtype=np.float32)
        self.register_buffer("mean_x", torch.tensor(mean_x, dtype=torch.float32))

        # Optional ICD-10 embedding (extension)
        if embed_dim > 0 and vocab_size > 0:
            self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode="mean", padding_idx=0)
        else:
            self.embedding = None

        # Decay networks: scalar gap → per-feature decay rates
        self.decay_x = nn.Linear(1, input_dim)
        self.decay_h = nn.Linear(1, hidden_dim)

        # GRU cell input: [x_hat | mask | sex_male | time_since_index | (embed)]
        gru_input_dim = input_dim + input_dim + 2 + embed_dim
        self.gru_cell = nn.GRUCell(gru_input_dim, hidden_dim)
        self.dropout  = nn.Dropout(dropout)
        self.linear   = nn.Linear(hidden_dim, 1)

    def forward(
        self,
        x,               # (B, T, D)  padded lab values
        mask,            # (B, T, D)  observation masks
        deltas,          # (B, T)     inter-observation gaps in years
        lengths,         # (B,)       true sequence lengths
        sex_male,        # (B,)
        time_since_index=None,  # (B, T)  optional
        flat_codes=None,        # EmbeddingBag input  (extension)
        offsets=None,           # EmbeddingBag offsets (extension)
    ):
        batch_size, max_len, _ = x.shape
        device = x.device

        # ICD embedding (extension only)
        if self.embedding is not None and flat_codes is not None:
            bag_embeds = self.embedding(flat_codes, offsets).view(batch_size, max_len, -1)
        else:
            bag_embeds = None

        h_t    = torch.zeros(batch_size, self.hidden_dim, device=device)
        x_last = self.mean_x.unsqueeze(0).repeat(batch_size, 1)  # (B, D)
        outputs  = []
        h_states = []

        for t in range(max_len):
            delta_t = deltas[:, t:t+1]   # (B, 1)
            x_t     = x[:, t, :]         # (B, D)  — 0 where lab not observed
            m_t     = mask[:, t, :]      # (B, D)

            # ── TODO 1 ── Lab-value decay ─────────────────────────────────────
            # Compute gamma_x, the per-feature decay factor for lab values.
            # Shape: (B, D)
            #
            # Hint: use self.decay_x (a Linear layer) to map delta_t to a rate,
            #       then ensure the result lies in (0, 1].
            #       Read Che et al. (2018), Eq. 3 — what function maps any real
            #       number to (0, 1] and equals 1 when the input is 0?
            #
            gamma_x = torch.exp(-torch.clamp(self.decay_x(delta_t), min=0.0))   # TODO 1

            # ── TODO 2 ── Hidden-state decay ──────────────────────────────────
            # Compute gamma_h, the decay factor for the hidden state.
            # Shape: (B, hidden_dim)
            #
            # Hint: same structure as TODO 1 but applied to the hidden state
            #       using self.decay_h. See Che et al. (2018), Eq. 4.
            #
            gamma_h = gamma_h = torch.exp(-torch.clamp(self.decay_h(delta_t), min=0.0))   # TODO 2

            # ── TODO 3 ── Impute missing values ───────────────────────────────
            # Build x_hat, the imputed lab vector the GRU cell receives.
            # Shape: (B, D)
            #
            # If m_t = 1 (observed): use the real value x_t unchanged.
            # If m_t = 0 (missing):  the last observed value has been decaying
            #                        since it was measured; at some point it should
            #                        revert fully to the population mean (self.mean_x).
            #                        gamma_x controls how much of the last value to keep.
            #
            # Hint: think in two parts — what do you use when a lab IS observed,
            #       and what do you use when it is NOT? Then combine with m_t.
            #       See Che et al. (2018), Eq. 5.
            #
            x_hat = m_t*x_t + (1.0-m_t)*(gamma_x*x_last+(1.0-gamma_x)*self.mean_x)   # TODO 3

            # Update x_last: remember the most recently *observed* value
            x_last = m_t * x_t + (1.0 - m_t) * x_last

            # ── TODO 4 ── Decay the hidden state ──────────────────────────────
            # Apply gamma_h to h_t *before* the GRU cell sees it.
            # Shape of h_t: unchanged, (B, hidden_dim)
            #
            # Hint: one line. The longer the gap since the last observation,
            #       the more the model should forget its current belief.
            #       See Che et al. (2018), Eq. 6.
            #
            h_t = gamma_h*h_t   # TODO 4

            # ── GRU cell update (provided — do not change) ────────────────────
            sex_t  = sex_male[:, None]                                  # (B, 1)
            time_t = (
                time_since_index[:, t:t+1]
                if time_since_index is not None
                else torch.zeros(batch_size, 1, device=device)
            )
            parts = [x_hat, m_t, sex_t, time_t]
            if bag_embeds is not None:
                parts.append(bag_embeds[:, t, :])
            input_t = torch.cat(parts, dim=-1)

            h_t = self.gru_cell(input_t, h_t)
            outputs.append(self.linear(self.dropout(h_t)))
            h_states.append(h_t)

        all_risks = torch.stack(outputs,  dim=1).squeeze(-1)   # (B, T)
        h_states  = torch.stack(h_states, dim=1)               # (B, T, H)

        # Risk score at each patient's last valid (non-padded) timestep
        last_idx  = (lengths - 1).clamp(min=0)
        batch_idx = torch.arange(batch_size, device=device)
        last_risk = all_risks[batch_idx, last_idx]              # (B,)

        return last_risk, all_risks, h_states


# Instantiate the model
model = TimeAwareGRUD(
    input_dim  = lab_dim,
    mean_x     = mean_x,
    hidden_dim = HIDDEN_DIM,
    embed_dim  = EMBED_DIM,
).to(device)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

TimeAwareGRUD(
  (decay_x): Linear(in_features=1, out_features=23, bias=True)
  (decay_h): Linear(in_features=1, out_features=32, bias=True)
  (gru_cell): GRUCell(48, 32)
  (dropout): Dropout(p=0.2, inplace=False)
  (linear): Linear(in_features=32, out_features=1, bias=True)
)

Parameters: 8,015


## 4. Loss — Cox partial likelihood *(TODO 6)*

The Cox proportional hazards model posits that the hazard for patient $i$ factors into
a shared baseline and a patient-specific risk score:

$$h(t \mid x_i) = h_0(t) \cdot \exp(\text{risk}(x_i))$$

The baseline $h_0(t)$ cancels out in the **partial likelihood** — only the risk scores
matter for training. Your task is to implement this loss from first principles.

**Read Cox (1972), JRSS-B, Section 4, then answer these questions before coding:**

1. At each event time $t_i$, what is the probability that *patient $i$* is the one
   who has the event, given everyone still at risk?
2. How does that probability simplify under the proportional-hazards assumption?
3. Why does sorting patients by **descending** duration turn the denominator into a
   prefix cumulative sum, enabling a single vectorised pass through the batch?

**Numerical tip:** `log(sum(exp(x)))` overflows for large `x`.
Subtract `max(x)` before exponentiating and add it back after the log
(log-sum-exp trick).

In [ ]:
def cox_ph_loss(
    risk_scores: torch.Tensor,   # (B,)  log-risk scores
    durations:   torch.Tensor,   # (B,)  time from landmark to event/censor
    events:      torch.Tensor,   # (B,)  1 = lung cancer, 0 = censored/other
    weights:     torch.Tensor | None = None,   # (B,)  optional importance weights
) -> torch.Tensor:
    """
    Negative Cox partial log-likelihood.

    Reference: Cox (1972) "Regression models and life tables", JRSS-B.
    """
    # ── TODO 6 ── Cox partial log-likelihood ──────────────────────────────────
    # Implement the negative Cox partial log-likelihood.
    #
    # Step 1 — Sort by descending duration.
    #   After sorting, the risk set for patient i is exactly the prefix [0..i].
    #   Apply the same permutation to risk_scores, events, and weights.
    #
    durations, indices = torch.sort(durations, descending=True)
    risk_scores = risk_scores[indices]
    events = events[indices]
    if weights is not None:
        weights = weights[indices]
    #
    # Step 2 — Compute log of the cumulative risk-set sum (log-sum-exp trick).
    #   Hint: torch.cumsum on exp(risk_scores), then torch.log.
    #   Subtract max(risk_scores) before exp and add it back after log to avoid overflow.
    #
    max_risk = torch.max(risk_scores)
    exp_risk = torch.exp(risk_scores - max_risk)
    cumsum_exp_risk = torch.cumsum(exp_risk, dim=0)
    log_cumsum_exp = torch.log(cumsum_exp_risk) + max_risk
    #
    # Step 3 — Per-patient log-likelihood contribution.
    #   For each patient: risk_score_i minus the log of the cumulative sum up to i.
    #   Only event patients (events == 1) contribute to the loss.
    #
    log_likelihood = risk_scores - log_cumsum_exp
    #
    # Step 4 — Normalise and negate.
    #   If weights are provided, weight each event patient's contribution
    #   by its sample weight before summing; normalise by the total weight.
    #   Otherwise normalise by the number of events.
    #   Return the negated scalar loss (we minimise, so we negate the log-likelihood).
    #
    if weights is not None:
        sum_weights = torch.sum(events * weights)
        if sum_weights == 0:
            loss = torch.tensor(0.0, device=risk_scores.device)
        else:
            loss = - torch.sum(log_likelihood * events * weights) / sum_weights
    else:
        sum_events = torch.sum(events)
        if sum_events == 0:
            loss = torch.tensor(0.0, device=risk_scores.device)
        else:
            loss = - torch.sum(log_likelihood * events) / sum_events
    return loss

## 5. Training epoch *(TODO 7)*

`run_raw_loss_epoch` runs one full pass over a DataLoader.
Pass `optimizer=None` for evaluation mode (no gradient updates).

Your task is to fill in the core of the loop: call the model, compute the loss,
and — when training — run the backward pass.

In [ ]:
def run_raw_loss_epoch(loader, model, device, optimizer=None):
    """
    One forward pass over the dataset.

    Returns (mean_loss, c_index).
    If optimizer is not None, also runs backward + step (training mode).

    Batch format: (labs, masks, deltas, lengths, sex_male, time_since_index,
                   durations, events, landmarks, weights, ids).
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_risks, all_dur, all_ev = [], [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            (labs, masks, deltas, lengths, sex_male,
             time_since_index, durations, events, landmarks, weights, ids) = batch

            labs             = labs.to(device)
            masks            = masks.to(device)
            deltas           = deltas.to(device)
            lengths          = lengths.to(device)
            sex_male         = sex_male.to(device)
            time_since_index = time_since_index.to(device)
            durations        = durations.to(device)
            events           = events.to(device)
            weights          = weights.to(device)

            # ── TODO 7 ── Forward pass, loss, and backward ───────────────────
            # 1. Call the model to get risk_scores.
            #    Signature: model(labs, masks, deltas, lengths, sex_male, time_since_index)
            #    Returns (last_risk, all_risks, h_states) — you only need the first.
            (risk_scores,_,_)= model(labs, masks, deltas, lengths, sex_male, time_since_index)
            #
            # 2. Compute the loss with cox_ph_loss(risk_scores, durations, events, weights).
            loss= cox_ph_loss(risk_scores, durations, events, weights)
            #
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            #

            total_loss += loss.item()
            all_risks.extend(risk_scores.detach().cpu().numpy())
            all_dur.extend(durations.cpu().numpy())
            all_ev.extend(events.cpu().numpy())

    mean_loss = total_loss / max(len(loader), 1)

    # ── TODO 8 ── C-index ────────────────────────────────────────────────────
    # Compute the concordance index (C-index) from the accumulated arrays:
    #   all_risks  — model log-risk scores  (higher = more risk)
    #   all_dur    — time from landmark to event or censoring
    #   all_ev     — event indicator (1 = lung cancer, 0 = censored)
    #
    # Use lifelines.utils.concordance_index(event_times, predicted_scores, event_observed).
    #
    # Think before you code:
    #   - concordance_index expects *higher* predicted_scores to mean *longer* survival.
    #     Our model outputs log-risk: higher risk → shorter survival.
    #     How should you transform all_risks before passing it?
    #   - The function expects event_observed as a boolean array.
    #
    ci = concordance_index(np.array(all_dur), -np.array(all_risks), np.array(all_ev, dtype=bool))
    return mean_loss, ci


## 6. Training

Run the cell below. After completing **TODO 5** you should see training loss
decrease each epoch and val C-index climb above 0.65 within ~10 epochs.

In [ ]:
from torch.utils.data import WeightedRandomSampler

# Event oversampling: give event patients EVENT_OVERSAMPLE× more draw probability
sample_weights = [EVENT_OVERSAMPLE if f == 1 else 1.0 for f in train_dataset.event_flags]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          collate_fn=raw_pad_collate)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=raw_pad_collate)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

best_val_loss = float("inf")
best_state    = None
no_improve    = 0

for epoch in range(1, MAX_EPOCHS + 1):

    # ── TODO 5 ── Training step ───────────────────────────────────────────────
    # Call run_raw_loss_epoch in training mode to run one full epoch and get
    # (train_loss, train_ci). Replace the two placeholder lines below.
    #
    # Hint: check the function signature above — what argument switches it
    #       between training and evaluation mode?
    #
    train_loss, train_ci = run_raw_loss_epoch(train_loader, model, device, optimizer)  # replace with the real call
    # ─────────────────────────────────────────────────────────────────────────

    val_loss, val_ci = run_raw_loss_epoch(val_loader, model, device, optimizer=None)
    scheduler.step(val_loss)

    print(f"Epoch {epoch:3d}/{MAX_EPOCHS}  "
          f"train_loss={train_loss:.4f}  train_ci={train_ci:.4f}  "
          f"val_loss={val_loss:.4f}  val_ci={val_ci:.4f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state    = copy.deepcopy(model.state_dict())
        no_improve    = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"  → early stopping at epoch {epoch}")
            break

if best_state is not None:
    model.load_state_dict(best_state)

print(f"\nBest val loss: {best_val_loss:.4f}")

Epoch   1/30  train_loss=4.8655  train_ci=0.6540  val_loss=3.9885  val_ci=0.6601
Epoch   2/30  train_loss=4.5468  train_ci=0.7452  val_loss=3.8859  val_ci=0.7715
Epoch   3/30  train_loss=4.3405  train_ci=0.8047  val_loss=3.8364  val_ci=0.7893
Epoch   4/30  train_loss=4.3031  train_ci=0.8095  val_loss=3.7723  val_ci=0.7924
Epoch   5/30  train_loss=4.2867  train_ci=0.8057  val_loss=3.7081  val_ci=0.8004
Epoch   6/30  train_loss=4.2239  train_ci=0.8145  val_loss=3.7034  val_ci=0.7976
Epoch   7/30  train_loss=4.2150  train_ci=0.8180  val_loss=3.7019  val_ci=0.8007
Epoch   8/30  train_loss=4.2080  train_ci=0.8138  val_loss=3.7389  val_ci=0.7987
Epoch   9/30  train_loss=4.1814  train_ci=0.8211  val_loss=3.7410  val_ci=0.7902
Epoch  10/30  train_loss=4.1863  train_ci=0.8193  val_loss=3.6950  val_ci=0.7997
Epoch  11/30  train_loss=4.1261  train_ci=0.8252  val_loss=3.7291  val_ci=0.7931
Epoch  12/30  train_loss=4.1312  train_ci=0.8236  val_loss=3.7452  val_ci=0.7968
Epoch  13/30  train_loss=4.1

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), OUT_DIR / "model.pt")
with open(OUT_DIR / "artifacts.pkl", "wb") as f:
    pickle.dump({
        "lab_dim":    lab_dim,
        "hidden_dim": HIDDEN_DIM,
        "embed_dim":  EMBED_DIM,
        "mean_x":     mean_x,
        "lab_codes":  lab_codes,
    }, f)
print(f"Saved → {OUT_DIR}/")

Saved → /content/drive/MyDrive/TFM/us-mulcia-2026-ramon/data_tmp/results/gru_d_student/


## 7. Evaluation

The cells below evaluate across **all landmarks (45–70)** and show a calibration
table at landmark 55. You need `results/gru_d_student/model.pt` from the training
cell above.

In [ ]:
def compute_baseline_hazard(risks, durations, events, horizon):
    """Breslow estimator: H0(horizon) from training data."""
    risks     = np.asarray(risks,     dtype=np.float64)
    durations = np.asarray(durations, dtype=np.float64)
    events    = np.asarray(events,    dtype=np.float64)

    max_r   = risks.max()
    shifted = risks - max_r
    H0 = 0.0
    for t in np.unique(durations[events == 1]):
        if t > horizon:
            continue
        d        = np.sum((durations == t) & (events == 1))
        risk_set = durations >= t
        denom    = np.sum(np.exp(shifted[risk_set]))
        if denom > 0:
            H0 += d / denom
    return H0, max_r


def collect_raw_outputs(loader, model, device):
    """
    Collect model outputs over a full loader.

    Returns (risks, durations, events, landmarks) as numpy arrays.
    """
    model.eval()
    risks, durs, evs, lms = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            (labs, masks, deltas, lengths, sex_male,
             time_since_index, durations, events, landmarks, weights, ids) = batch
            r, _, _ = model(
                labs.to(device), masks.to(device), deltas.to(device),
                lengths.to(device), sex_male.to(device), time_since_index.to(device),
            )
            risks.extend(r.cpu().numpy())
            durs.extend(durations.numpy())
            evs.extend(events.numpy())
            lms.extend(landmarks.numpy())
    return (np.array(risks), np.array(durs),
            np.array(evs, dtype=float), np.array(lms))

In [ ]:
# ── C-index across all landmarks ─────────────────────────────────────────────
landmark_times = list(range(45, 71))

print(f"{'Landmark':>9}  {'N':>6}  {'Events':>7}  {'C-index':>9}")
print("-" * 38)
ci_list = []
for lm in landmark_times:
    ds = RawSequenceDataset(
        sequences_val, cohort_val,
        landmark_ages=[float(lm)], index_age=INDEX_AGE,
        max_events=MAX_SEQ_LEN, age_weight_gamma=AGE_WEIGHT_GAMMA,
    )
    if not ds.samples:
        continue
    ld = DataLoader(ds, batch_size=512, shuffle=False, collate_fn=raw_pad_collate)
    risks, durs, evs, _ = collect_raw_outputs(ld, model, device)
    n_ev = int(evs.sum())
    if n_ev < 2:
        continue
    ci = concordance_index(durs, -risks, evs.astype(bool))
    ci_list.append(ci)
    print(f"{lm:>9}  {len(ds):>6}  {n_ev:>7}  {ci:>9.4f}")

print(f"\nMean C-index: {np.mean(ci_list):.4f}")

 Landmark       N   Events    C-index
--------------------------------------
       45    1237       54     0.7535
       46    1204       52     0.7583
       47    1184       52     0.7718
       48    1171       51     0.7656
       49    1151       50     0.7799
       50    1127       49     0.7764
       51    1112       47     0.7569
       52    1091       46     0.7807
       53    1072       44     0.7798
       54    1045       43     0.7917
       55    1028       43     0.7997
       56    1010       42     0.7968
       57     986       41     0.8026
       58     967       41     0.8140
       59     948       40     0.8133
       60     932       39     0.8322
       61     912       38     0.8316
       62     888       38     0.8568
       63     866       37     0.8728
       64     845       36     0.8930
       65     818       35     0.8792
       66     791       27     0.8763
       67     767       26     0.8776
       68     743       23     0.8600
       69  

In [ ]:
# ── Calibration table at landmark 55 ─────────────────────────────────────────
# HORIZON=5
ds_tr55 = RawSequenceDataset(
    sequences_train, cohort_train, [55.0], INDEX_AGE, MAX_SEQ_LEN, AGE_WEIGHT_GAMMA)
ds_vl55 = RawSequenceDataset(
    sequences_val,   cohort_val,   [55.0], INDEX_AGE, MAX_SEQ_LEN, AGE_WEIGHT_GAMMA)

ld_tr = DataLoader(ds_tr55, batch_size=512, shuffle=False, collate_fn=raw_pad_collate)
ld_vl = DataLoader(ds_vl55, batch_size=512, shuffle=False, collate_fn=raw_pad_collate)

train_risks, train_durs, train_evs, _ = collect_raw_outputs(ld_tr, model, device)
val_risks,   val_durs,   val_evs,   _ = collect_raw_outputs(ld_vl, model, device)

H0, max_r = compute_baseline_hazard(train_risks, train_durs, train_evs, HORIZON)
pred_prob  = 1.0 - np.exp(-H0 * np.exp(val_risks - max_r))

print(f"Mean predicted 5-yr risk : {pred_prob.mean():.4f}")
val_evs_horizon = ((val_durs <= HORIZON) & (val_evs == 1)).astype(float)
print(f"Observed 5-yr event rate : {val_evs_horizon.mean():.4f}")
# print(f"Observed event rate      : {val_evs.mean():.4f}")
print(f"E/O ratio                : {pred_prob.mean() / val_evs_horizon.mean():.3f}  (ideal = 1.0)")
# print(f"E/O ratio                : {pred_prob.mean() / val_evs.mean():.3f}  (ideal = 1.0)")
print()

# # Calibration by decile
# order = np.argsort(pred_prob)
# bins  = np.array_split(order, 10)
# print(f"{'Decile':>7}  {'Mean pred':>10}  {'Obs rate':>9}  {'N':>5}  {'Events':>7}")
# print("-" * 46)
# for i, idx in enumerate(bins):
#     print(f"{i+1:>7}  {pred_prob[idx].mean():>10.4f}  "
#           f"{val_evs[idx].mean():>9.4f}  {len(idx):>5}  {int(val_evs[idx].sum()):>7}")

order = np.argsort(pred_prob)
bins  = np.array_split(order, 10)
print(f"{'Decile':>7}  {'Mean pred':>10}  {'Obs rate':>9}  {'N':>5}  {'Events':>7}")
print("-" * 46)
for i, idx in enumerate(bins):
    print(f"{i+1:>7}  {pred_prob[idx].mean():>10.4f}  "
          f"{val_evs_horizon[idx].mean():>9.4f}  {len(idx):>5}  {int(val_evs_horizon[idx].sum()):>7}")

Mean predicted 5-yr risk : 0.0058
Observed 5-yr event rate : 0.0039
E/O ratio                : 1.486  (ideal = 1.0)

 Decile   Mean pred   Obs rate      N   Events
----------------------------------------------
      1      0.0001     0.0000    103        0
      2      0.0002     0.0000    103        0
      3      0.0005     0.0000    103        0
      4      0.0008     0.0000    103        0
      5      0.0014     0.0000    103        0
      6      0.0018     0.0000    103        0
      7      0.0023     0.0097    103        1
      8      0.0033     0.0000    103        0
      9      0.0100     0.0098    102        1
     10      0.0378     0.0196    102        2


En este último código he ajustado la tasa de eventos para que sea coherente con el horizonte a 5 años. Inicialmente se estaba comparando las probabilidades medias de sifrir el evento a 5 años contra la tasa de evento total. En la modificación se ajusta la tasa de evento para que se calcule solo con los eventos dentro del horizonte

## 8. Extensions

Once the basic version works, try these:

1. **All landmarks**: change `[LANDMARK_AGE]` to `list(range(45, 71))` in the
   `RawSequenceDataset` calls and retrain. Does the model generalise?

2. **ICD-10 embedding**: set `EMBED_DIM = 16`. Build a vocabulary mapping each ICD-10
   code to an integer index, then use `nn.EmbeddingBag` to embed each timestep's code
   bag into a fixed-size vector. Pass the result to the GRU cell alongside the labs.
   Compare C-index to the lab-only version.

3. **Competing risks**: how does the loss change when there are two possible outcomes
   (lung cancer vs. other death)? Look up the Fine-Gray subdistribution hazard model
   and think about how the risk set changes for competing events.

4. **Compare to Cox**: load `results/predictions/cox_locf/discovery_val/all.parquet`
   and compare C-index at landmark 55. How much does the sequence model add?

## 9. Future Work — Handling the Sequence Truncation Problem

The current model truncates each patient's timeline to the **200 most recent timesteps**
before the landmark (`MAX_SEQ_LEN = 200`). Because all event types — ICD diagnoses, lab A,
lab B — are merged into a single time-sorted list, this silently drops early events for
patients with dense histories: a diabetes diagnosis at age 18 or the first measurement
of a rarely-drawn lab can vanish entirely.

Below are four progressively more principled ways to fix this.

---

### Option 1 — Raise `MAX_SEQ_LEN` *(trivial)*

```python
MAX_SEQ_LEN = 500   # or 0 to disable truncation
```

**Justification:** On this Synthea dataset, almost no patient exceeds 200 events before
age 55, so the fix costs GPU memory but rarely changes the result. On real dense EHR data
(hospitalisations, frequent ICU vitals) the benefit would be larger. It is a useful
baseline to establish that truncation is not hurting the current numbers before investing
in a more principled fix.

---

### Option 2 — Keep first + last *(2-line change)*

```python
if max_events and len(times_s) > max_events:
    # always anchor on the first timestep, then take the (max_events-1) most recent
    idx = np.concatenate([[0], np.arange(len(times_s) - (max_events - 1), len(times_s))])
    times_s, labs_s, masks_s = times_s[idx], labs_s[idx], masks_s[idx]
```

**Justification:** The very first timestep often carries disproportionate information
(cohort-entry diagnoses, baseline labs). Preserving it as an anchor means the GRU-D decay
has a correct reference point for the oldest observation of every sparse lab, and early
chronic comorbidities (asthma, COPD) remain visible. Cost: one extra row per patient,
negligible memory impact.

---

### Option 3 — Separate ICD history from the truncated lab window *(~30 lines)*

Rather than including ICD codes in the merged timeline (where they compete for the 200
slots), summarise the **full** pre-landmark ICD history as a single fixed-size vector and
inject it as a static side-input alongside `sex_male`:

```python
# In the dataset: collect all codes seen before the landmark (no length limit)
all_codes_before_lm = [c for t, codes in zip(seq["times"], seq["codes"])
                        if t <= lm for c in codes]

# In the model __init__:
self.icd_embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode="mean")
self.linear = nn.Linear(hidden_dim + embed_dim, 1)   # wider head

# In forward: look up once, concatenate with h_T
icd_vec  = self.icd_embedding(flat_codes, offsets)   # (B, embed_dim)
last_risk = self.linear(torch.cat([h_T, icd_vec], dim=-1))
```

**Justification:** Diagnoses are cumulative facts, not time series — knowing a patient
had COPD at 30 is still relevant at 55 regardless of how many lab draws occurred in
between. Decoupling them from the sequential window means the GRU-D recurrence focuses
entirely on the temporal lab trajectory while the full comorbidity burden is always
represented. This matches common clinical NLP architectures (e.g. RETAIN, Med-BERT).

---

### Option 4 — Time-decayed ICD embedding *(Option 3 + temporal weighting)*

Extend Option 3 so that **recent diagnoses contribute more than old ones**, using the
same exponential-decay philosophy as GRU-D itself:

$$\mathbf{v}_{\text{ICD}} = \sum_{i} \exp\!\left(-\lambda \cdot (\text{landmark} - t_i)\right) \cdot \mathbf{e}(c_i)$$

where $t_i$ is the age at diagnosis, $c_i$ the ICD code, and $\lambda > 0$ a learnable
scalar (initialised to a small positive value so recent diagnoses dominate early in
training).

```python
class TimedICDEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.log_lambda = nn.Parameter(torch.tensor(0.0))   # exp → always positive

    def forward(self, codes, ages, landmark_age):
        # codes: (B, K)  ages: (B, K)  — padded with 0 / landmark_age
```

**Justification:** A static EmbeddingBag (Option 3) treats a COPD diagnosis from age 20
and one from age 54 identically. Temporal weighting lets the model learn that a recent
exacerbation is more prognostically relevant than a distant resolved episode — a medically
sensible inductive bias. The decay rate $\lambda$ is shared across codes but could be
made code-specific for a richer model. This approach is publishable as a lightweight
alternative to full transformer-based clinical sequence models.
